# Revision results and manuscript tables

Consume explicitly selected saved full-volume CSVs from training, K-Fold, ablation, model comparison or offline evaluation. This notebook does not retrain models or repeat inference. Use the **rnet** kernel.

The biological animal is the statistical unit. Repeated model predictions are averaged per scan, then longitudinal scans per animal. This estimates performance and volume agreement for each animal's mean scan measurement; it does not model trajectories or training-seed variability. Source metadata and raw annotation/empty-mask statuses remain auditable.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT: Path = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/experiments/config.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from IPython.display import display
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "vscode"

## Explicit result sources and analysis settings

For one run, use `regions/full_volume_metrics.csv` and supply its variant name and training seed. For a study, use its top-level `full_volume_metrics.csv` with `name=None` to retain existing variant/seed columns. Offline evaluation saves the same long-form contract. Do not load both a study CSV and its individual run CSVs: duplicate measures are rejected.

External nnU-Net results can enter as an explicit CSV with canonical `Subject`, `Scan`, `Dataset`, `Region`, `Metric`, `Value`, `Unit`, `Status` columns and explicit variant/seed/fold/role metadata. Use the same biological IDs, annotations, region semantics and physical units. Missing annotations must be NaN with `Status="unavailable"`.

Validation roles are excluded by default. To summarize historical rat CV, explicitly select `roles=("validation",)` and describe those results as checkpoint-selection validation. Separate cohorts and anatomical regions are never pooled implicitly.

In [ ]:
from src.evaluation.revision import MetricSource, RevisionConfig, export_revision, revision_report

METRIC_SOURCES: list[MetricSource] = [
    # MetricSource(path=REPO_ROOT / "results/comparisons/d1_3drnet_vs_unet/full_volume_metrics.csv"),
    # MetricSource(path=Path(".../regions/full_volume_metrics.csv"), name="3drnet", seed=42),
]
analysis = RevisionConfig(
    roles=("test", "holdout", "external"), metrics=("Dice", "HD95", "ASSD", "AVE"),
    bootstrap_samples=2000, confidence=0.95, seed=42,
)
MODEL_COMPARISON: tuple[str, str] | None = None  # ("3drnet", "unet")
RUN_ANALYSIS: bool = False
EXPORT_TABLES: bool = False
OUTPUT_DIR: Path = REPO_ROOT / "outputs" / "revision" / "selected_results"
CLASSIFIER_FILES: tuple[Path, ...] = ()  # Saved run summary/study/checkpoint classifier_predictions.csv.
RUN_CLASSIFICATION: bool = False
EXPORT_CLASSIFICATION: bool = False
CLASSIFIER_OUTPUT_DIR: Path = OUTPUT_DIR / "classification"
COST_FILES: tuple[Path, ...] = ()  # Study timings.csv or per-run summary/fold_timings.csv

print("Selected metric sources:", len(METRIC_SOURCES))
display(pd.DataFrame({"File": [str(source.path) for source in METRIC_SOURCES]}))

## Animal-aware summaries and coverage

Intervals resample animals after the prespecified scan/model aggregation. They condition on the supplied trained models. Missing annotations, undefined surface distances and invalid values are not zero-filled. Finite both-empty Dice values retain the metric's documented convention; inspect their status counts. The summary reports the number of animals with finite measurements for each region/metric.

In [ ]:
report = None
if RUN_ANALYSIS:
    report = revision_report(sources=METRIC_SOURCES, config=analysis, comparison=MODEL_COMPARISON)
    display(report.statuses)
    display(report.summary)
    display(report.animals)
else:
    print("Select saved result files, then set RUN_ANALYSIS = True.")

## Segmentation performance

Plot one descriptive value per animal. Longitudinal scan counts remain visible; multiple seeds/fold predictions are not shown as additional animals. Use dataset/role filters to compare the intended cohort.

In [ ]:
if report is not None:
    dice = report.animals.loc[report.animals["Metric"] == "Dice"]
    if not dice.empty:
        figure = px.box(
            dice, x="Variant", y="Value", color="Variant", facet_col="Region",
            facet_row="Dataset", points="all", hover_data=["Subject", "DatasetRole", "Scans"],
            title="Animal mean full-volume Dice",
        )
        figure.show()

## Volume agreement: MAE, Bland–Altman, ICC(A,1), CCC

Reference and prediction volumes are paired before averaging. Genuine zero reference volumes remain valid; missing annotations are excluded. This is agreement for **animal mean scan volumes**, not longitudinal limits of agreement. At least three independent animals are required. Check the number of annotated scans contributing to each animal.

Normal-theory limits of agreement assume approximately normal differences and homogeneous variance. Inspect the difference/Q-Q diagnostics and proportional-bias estimate. Bootstrap confidence intervals describe sampling uncertainty; they are distinct from limits of agreement. Existing paired and variance diagnostics are exploratory and do not establish interchangeability.

In [ ]:
from src.evaluation.statistics import (
    AgreementConfig, bland_altman_figure, calculate_agreement, difference_distribution_figure,
)

AGREEMENT_VARIANT: str = "3drnet"
AGREEMENT_DATASET: str = "D1"
AGREEMENT_REGION: str = "Lesion"
AGREEMENT_ROLE: str = "test"

if report is not None:
    display(report.agreement)
    if not report.volumes.empty:
        pairs = report.volumes.loc[
            (report.volumes["Variant"] == AGREEMENT_VARIANT)
            & (report.volumes["Dataset"] == AGREEMENT_DATASET)
            & (report.volumes["Region"] == AGREEMENT_REGION)
            & (report.volumes["DatasetRole"] == AGREEMENT_ROLE)
        ]
        display(pairs)
        if len(pairs) >= 3:
            agreement_settings = AgreementConfig(
                unit="mm3", bootstrap_samples=analysis.bootstrap_samples,
                confidence=analysis.confidence, seed=analysis.seed,
            )
            agreement = calculate_agreement(pairs, agreement_settings)
            bland_altman_figure(pairs, agreement, agreement_settings).show()
            difference_distribution_figure(pairs, agreement_settings).show()
        else:
            print("Choose an agreement group with at least three independent animals.")

## Optional paired architectural comparison

Set `MODEL_COMPARISON` before analysis. Each comparison uses scans with finite measurements from both models, then pairs animal means. Positive differences mean candidate minus reference; higher Dice and lower distances/errors have different preferred directions. Review paired animal counts and common-scan coverage alongside the separate model summaries.

Mean-difference intervals resample paired animals. Exploratory Wilcoxon p-values receive Holm adjustment over all tested dataset/role/region/metric groups returned in this analysis. No test is chosen using normality-test results. No comparison is performed with fewer than three paired animals. Hyperparameters and cohorts must have been selected independently of these evaluation results.

In [ ]:
if report is not None:
    display(report.comparisons)

## Auxiliary classification and healthy lesion false positives

Select complete saved classifier decision tables. SHAM=0 and TBI=1. Scan predictions retain the historical majority of patch decisions (>0.5; ties SHAM); the recorded positive patch fraction is not a calibrated probability. Longitudinal scan decisions are combined by majority per animal, again with ties SHAM. Rates and exact 95% intervals count animals, not scans or seeds. Disjoint outer holdouts also have a pooled row; repeated test/external predictions stay per fold.

Classifier decisions never gate lesion segmentation. SHAM lesion volumes and the fraction of healthy animals with any nonzero lesion prediction are separate diagnostics. These diagnostics also apply to no-classifier models. Validation remains checkpoint-selection performance. Older results without classifier records require fresh checkpoint evaluation.


In [ ]:
from src.evaluation.classification import (
    classification_report, load_classifier_predictions, export_classification_tables,
)

classifier_results = None
classifier_scans = pd.DataFrame()
if RUN_CLASSIFICATION:
    classifier_scans = load_classifier_predictions(CLASSIFIER_FILES)
    if classifier_scans.empty:
        raise ValueError("Select saved classifier_predictions.csv files first")
    classifier_scans = classifier_scans.loc[classifier_scans["DatasetRole"].isin(analysis.roles)]
    if classifier_scans.empty:
        raise ValueError("No classifier decisions match the selected cohort roles")
    classifier_results = classification_report(classifier_scans, confidence=analysis.confidence)
    display(classifier_results.summary)
    display(classifier_results.animals)
if report is not None:
    display(report.sham_lesion_summary)
    display(report.sham_lesion_animals)


In [ ]:
import plotly.graph_objects as go

CLASSIFIER_VARIANT: str = "3drnet"
CLASSIFIER_SEED: int = 42
CLASSIFIER_ROLE: str = "test"
CLASSIFIER_SCOPE: str = "fold"  # pooled_holdout uses Fold=-1.
CLASSIFIER_FOLD: int = 0

if classifier_results is not None:
    confusion = classifier_results.confusion
    selected = confusion.loc[
        (confusion["Variant"] == CLASSIFIER_VARIANT)
        & (confusion["Seed"] == CLASSIFIER_SEED)
        & (confusion["DatasetRole"] == CLASSIFIER_ROLE)
        & (confusion["Scope"] == CLASSIFIER_SCOPE)
        & (confusion["Fold"] == CLASSIFIER_FOLD)
    ]
    if selected.empty:
        print("Choose a variant/seed/role/scope/fold from the summary above.")
    else:
        matrix = selected.pivot(index="Reference", columns="Prediction", values="Count")
        counts = matrix.to_numpy()
        heatmap = go.Heatmap(
            z=counts, text=counts, texttemplate="%{text}",
            x=["Predicted SHAM", "Predicted TBI"],
            y=["Reference SHAM", "Reference TBI"], colorscale="Blues",
        )
        figure = go.Figure(heatmap)
        figure.update_layout(title="Animal-level classifier confusion matrix")
        figure.show(renderer="vscode")

if EXPORT_CLASSIFICATION:
    if classifier_results is None:
        raise ValueError("Run classifier analysis before exporting")
    CLASSIFIER_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
    source_metrics = pd.DataFrame()
    if report is not None:
        source_metrics = report.raw.loc[report.raw["DatasetRole"].isin(analysis.roles)]
    export_classification_tables(
        classifier_scans, source_metrics, CLASSIFIER_OUTPUT_DIR, confidence=analysis.confidence,
    )
    print("Classifier and SHAM tables saved to:", CLASSIFIER_OUTPUT_DIR)


## Computational costs

Choose explicit timing tables. Keep model, seed, fold, device and precision provenance with comparisons. Inspect actual epochs and parameter counts as well as elapsed training/evaluation times. Per-volume inference tables separately record preprocessing and reconstruction durations; those times exclude warm-up but do not isolate GPU kernels.

In [ ]:
for cost_file in COST_FILES:
    print(cost_file)
    display(pd.read_csv(cost_file))

## Export auditable manuscript tables

`EXPORT_TABLES=True` writes raw scan metrics, animal metrics, uncertainty summaries, status counts, paired volumes, agreement and optional model comparisons into a fresh folder, together with source paths and analysis settings. Existing folders are rejected. CSVs retain full numeric precision and can be formatted separately for the manuscript. No interpolation sensitivity analysis is included.

In [ ]:
if EXPORT_TABLES:
    if report is None:
        raise ValueError("Run the analysis before exporting")
    saved_folder = export_revision(result=report, output_dir=OUTPUT_DIR, config=analysis)
    print("Tables saved to:", saved_folder)